# MOM 3D Plate - Mesh Generation

We describe first the use of [GMSH](gmsh.info) for the generation of the geometry and the generation of the geometry of the 3D plate.

## Import Packages

In [4]:
try
    using Gmsh: Gmsh, gmsh
catch
    using gmsh
end 

using LinearAlgebra
using Test 

using Plots

## Section 1: Introduction 

We fist use [GMSH OpenCASCADE CAD kernel functions](https://gmsh.info/doc/texinfo/gmsh.html#Namespace-gmsh_002fmodel_002focc) to generate the geometry of the plate of dimension 1 meter in $x$-direction by 1 meter in $y$-direction by 0.1 meter in $z$-direction.  

We subsequently use [GMSH](gmsh.info) to generate a mesh on the plate. We set <i>Mesh.Algorithm3D = 1</i> to ensure that the mesh consists of tetrahedral elements only. 

More later. 

## Section 2: Geometry and Mesh Generation 

<b>Exercises</b>
1. change the dimensions of the mesh and generate the mesh again;
2. change the number of elements in the mesh;
3. view the element and node tags in the graphical users interface; 

In [5]:
Gmsh.finalize()

In [6]:
#..1/8: initialize gmsh 
should_finalize = Gmsh.initialize()

#..2/8: set GMSH global options 
gmsh.option.set_number("General.Verbosity",3 )         # make more verbose
#....force the 3D algorithm to Tetrahedra (Default is 1: Delaunay) 4 (Frontal), 7 (MMG3D)
gmsh.option.setNumber("Mesh.Algorithm3D", 1)
gmsh.option.setNumber("Mesh.MeshSizeFromCurvature",2)
gmsh.option.setNumber("Mesh.MeshSizeMin",0.2)          # set mesh density 
gmsh.model.mesh.recombine()

#..3/8: generate geometry (unit square 0 <= x <= 1 and 0 <= y <= 1)
volume_tag = gmsh.model.occ.add_box(0,0,0,1,1,0.1,1)

#..4/8: synchronize
gmsh.model.occ.synchronize()

#..6/8: generate mesh
gmsh.model.mesh.generate(3)

#..7/8: write to file
if (true) gmsh.write("my_box.msh") end
if (true) gmsh.fltk.run() end

#..8/8: finalize
should_finalize && Gmsh.finalize(); 

-------------------------------------------------------
Version       : 4.13.1
License       : GNU General Public License
Build OS      : MacOSX-sdk
Build date    : 19700101
Build host    : amdci7.julia.csail.mit.edu
Build options : 64Bit ALGLIB[contrib] ANN[contrib] Bamg Blossom Cairo DIntegration Dlopen DomHex Eigen[contrib] Fltk GMP Gmm[contrib] Hxt Jpeg Kbipack MathEx[contrib] Mesh Metis[contrib] Mmg Mpeg Netgen Nii2mesh ONELAB ONELABMetamodel OpenCASCADE OpenCASCADE-CAF OpenGL OpenMP OptHom Parser Plugins Png Post QuadMeshingTools QuadTri Solver TetGen/BR TinyXML2[contrib] Untangle Voro++[contrib] WinslowUntangler Zlib
FLTK version  : 1.3.8
OCC version   : 7.7.2
Packaged by   : root
Web site      : https://gmsh.info
Issue tracker : https://gitlab.onelab.info/gmsh/gmsh/issues
-------------------------------------------------------


2026-08-21 17:05:05.641 julia[58757:10404829] +[IMKClient subclass]: chose IMKClient_Modern
2026-08-21 17:05:05.641 julia[58757:10404829] +[IMKInputSession subclass]: chose IMKInputSession_Modern


## Section 3: Read Mesh from File and Loop over the Elements   

<b>Exercises</b>
1. extend code below to recover faces, edges and node labels belonging to an element; 

In [7]:
#..1/7: Finalize gmsh
should_finalize = Gmsh.initialize()

#..2/7: Read mesh from file
gmsh.open("my_box.msh")

#..3/7: Get the line mesh entity (this part can be skipped) 
# Get all the elementary entities in the model, as a vector of (dimension, tag) pairs
# In case of line tutorial, return dim = 1 and tag = 1 
# or entities[1] = (1,1)
entities = gmsh.model.getEntities(1)

#..4/7: Get elements 
dim = 3 
elemTypes, elemTags, elemNodeTags = gmsh.model.mesh.getElements(dim, 1)

#..5/7: Get nodes  
nodeTags, node_coord, _ = gmsh.model.mesh.getNodes()
xnode_coord = node_coord[1:3:end];  

#..6/7: Loop over elements - requires more explanation 
quad = 4
for (i,elemtag) in enumerate(elemTags[1])
   offset = quad*(i-1)
   idx = offset+1:offset+quad   
   inode = elemNodeTags[1][idx]
   xnode = node_coord[3*(inode.-1).+1];
   ynode = node_coord[3*(inode.-1).+2];
   znode = node_coord[3*(inode.-1).+3];
   if(false)
     println("  elemtag = ",elemtag)  
     println("    elemnode1tag = ",inode[1]) 
     println("      xnode1 = ",xnode[1])
     println("    elemnode2tag = ",inode[2]) 
     println("      xnode2 = ",xnode[2])
     println("    elemnode1tag = ",inode[3]) 
     println("      xnode3 = ",xnode[3])
     println("    elemnode2tag = ",inode[4]) 
     println("      xnode4 = ",xnode[4])
    end 
end 

#..7/7: finalize gmsh 
should_finalize && Gmsh.finalize() 

Info    : Reading 'my_box.msh'...
Info    : 27 entities
Info    : 108 nodes
Info    : 526 elements
Info    : Done reading 'my_box.msh'


## Section 4: Loop over Elements and Assemble Matrices and Vectors  

In [8]:
#..1/7: Finalize gmsh
should_finalize = Gmsh.initialize()

#..2/7: Read mesh from file
gmsh.open("my_box.msh")

#..initialize matrix 
node_tags, _, _ = gmsh.model.mesh.getNodes()
nnodes = length(node_tags)
A = zeros(nnodes,nnodes)

#..3/7: Get the line mesh entity (this part can be skipped) 
# Get all the elementary entities in the model, as a vector of (dimension, tag) pairs
# In case of line tutorial, return dim = 1 and tag = 1 
# or entities[1] = (1,1)
entities = gmsh.model.getEntities(1)

#..4/7: Get elements 
dim = 3 
elemTypes, elemTags, elemNodeTags = gmsh.model.mesh.getElements(dim, 1)

#..5/7: Get nodes  
nodeTags, node_coord, _ = gmsh.model.mesh.getNodes()
xnode_coord = node_coord[1:3:end];  

#..6/7: Loop over elements - requires more explanation 
quad = 4
for (i,elemtag) in enumerate(elemTags[1])
   offset = quad*(i-1)
   idx = offset+1:offset+quad   
   inode = elemNodeTags[1][idx]
   xnode = node_coord[3*(inode.-1).+1];
   ynode = node_coord[3*(inode.-1).+2];
   znode = node_coord[3*(inode.-1).+3];
   Aloc = ones(4,4)
   A[inode,inode] += Aloc 
   if(false)
     println("  elemtag = ",elemtag)  
     println("    elemnode1tag = ",inode[1]) 
     println("      xnode1 = ",xnode[1])
     println("    elemnode2tag = ",inode[2]) 
     println("      xnode2 = ",xnode[2])
     println("    elemnode1tag = ",inode[3]) 
     println("      xnode3 = ",xnode[3])
     println("    elemnode2tag = ",inode[4]) 
     println("      xnode4 = ",xnode[4])
    end 
end 

#..7/7: finalize gmsh 
should_finalize && Gmsh.finalize() 

Info    : Reading 'my_box.msh'...
Info    : 27 entities
Info    : 108 nodes
Info    : 526 elements
Info    : Done reading 'my_box.msh'
